## Tool

Tools are pairing of 
1. A schema - often comprises of function name (tool name), description and function args
2. A function or coroutine to execute

In [1]:
from langchain.chat_models import init_chat_model

model = init_chat_model("openai/gpt-oss-120b", model_provider="groq")
response = model.invoke("How are you doing?")
response

AIMessage(content="I'm doing great, thanks for asking! How can I assist you today?", additional_kwargs={'reasoning_content': 'The user asks a casual question: "How are you doing?" The assistant should respond politely, perhaps ask about the user, be friendly. No policy violation.'}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 76, 'total_tokens': 133, 'completion_time': 0.119733156, 'completion_tokens_details': {'reasoning_tokens': 33}, 'prompt_time': 0.002995716, 'prompt_tokens_details': None, 'queue_time': 3.795184566, 'total_time': 0.122728872}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4a7bb72c24', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a116bc-f025-7ae3-96d9-9979cbae8f45-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 57, 'total_tokens': 133, 'output_token_details': {'reasoning': 33}})

In [2]:
from langchain.tools import tool

@tool(description="Get the current weather for a given location")
def get_current_weather(location: str) -> str:
    # For the purpose of this example, we'll return a hardcoded response
    return f"It is cloudy in {location}"

@tool(description="Get the current temperature for a given location")
def get_current_temperature(location: str) -> str:
    # For the purpose of this example, we'll return a hardcoded response
    return f"The current temperature is 23°F"

model_with_tools = model.bind_tools([get_current_weather,get_current_temperature])
response = model_with_tools.invoke("What is the current weather and temperature in New York?")

print(response)


content='' additional_kwargs={'reasoning_content': "User wants current weather and temperature in New York. We have functions for weather and temperature. Could call get_current_weather and get_current_temperature. Probably combine. Let's call both.", 'tool_calls': [{'id': 'fc_66dc4bbf-7020-4f26-b21d-6f4b775e6788', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_current_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 159, 'total_tokens': 224, 'completion_time': 0.137200228, 'completion_tokens_details': {'reasoning_tokens': 36}, 'prompt_time': 0.044953954, 'prompt_tokens_details': None, 'queue_time': 0.36913057, 'total_time': 0.182154182}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4140daa9c2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a116bd-0359-7442-b024-2b982fbe572d-0' tool_calls=[{'name': 'get_current_weather

## Tool Execution Loop

When multiple tools are registered with a model, the model may choose to call them in a single invokation or in multiple invokations. As long as the tool calls are pending, the model considers that it has not reached the end of job execution. In the following example, the model - "openai/gpt-oss-120b" has a tendency to invoke tool calls one by one, hence we are running a loop on tool_calls.

In [15]:
tools = {
    "get_current_weather": get_current_weather,
    "get_current_temperature": get_current_temperature,
}

messages = [{"role": "user", "content": "What is the weather and temperature in New York?"}]
ai_message = model_with_tools.invoke(messages)

print(f"First AI Message: {ai_message}")
print("========")

while ai_message.tool_calls:
    messages.append(ai_message)
    
    for tool_call in ai_message.tool_calls:
        tool_result = tools[tool_call["name"]].invoke(tool_call)
        print(f"Tool result: {tool_result}")
        print("========")
        messages.append(tool_result)
    
    
    ai_message = model_with_tools.invoke(messages)
    print(f"AI Message: {ai_message.additional_kwargs.get('reasoning_content')}")


print("========")
print(f"Final Response: {ai_message}")    

# print("messages============")

# for message in messages:
#     print(message)
    
# print("end============")
# print(final_response)

First AI Message: content='' additional_kwargs={'reasoning_content': "User wants weather and temperature in New York. We have function to get current weather and temperature. Likely need both. We can call get_current_weather and get_current_temperature. Let's call weather first.", 'tool_calls': [{'id': 'fc_4502ac27-e0cc-42a0-b1b5-2b04a2d6beda', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_current_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 158, 'total_tokens': 228, 'completion_time': 0.148412187, 'completion_tokens_details': {'reasoning_tokens': 41}, 'prompt_time': 0.064105878, 'prompt_tokens_details': None, 'queue_time': 0.646550016, 'total_time': 0.212518065}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_9241e9962b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a116d2-7b49-7510-adc2-eca954f8c8a5-0' tool_calls